# 01 — Bronze Ingestion

Ingests raw source data into Bronze Delta tables.

**HubSpot** is handled here using an API-style notebook (cursor pagination + tenacity retry).  
**Dynamics 365** and **Paradigm** are handled by Copy Job activities in the Data Pipeline.

Bronze tables preserve source data exactly as received. No type casting, no cleaning. Silver does that.

## Demo vs Production

| Source | Demo | Production |
|---|---|---|
| HubSpot | Reads JSON from `Files/raw/hubspot/` | Replace `_fetch_*` body with `requests.get` to `api.hubapi.com/crm/v3` + bearer auth |
| Dynamics 365 | Copy Job reads CSV from `Files/raw/dynamics365/` | Copy Job uses native Dataverse connector |
| Paradigm | Copy Job reads CSV from `Files/raw/paradigm/` | Copy Job reads from scheduled SFTP/blob exports |

The code shape (pagination loop, retry decorator, auth header stub) mirrors production. Only the data source swaps.

In [ ]:
import json
from typing import Iterator

from tenacity import retry, stop_after_attempt, wait_exponential

LAKEHOUSE_RAW = "Files/raw"
BRONZE = "bronze"


## HubSpot

Three entities ingested: `marketing_sources`, `contacts`, `activities`.

In [ ]:
# PRODUCTION: inject bearer token from Key Vault
# _HS_HEADERS = {
#     "Authorization": f"Bearer {notebookutils.credentials.getSecret('kv-jmc', 'hubspot-token')}"
# }


@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def _hs_get(path: str) -> list[dict]:
    # PRODUCTION: replace with requests.get(f"https://api.hubapi.com/{path}", headers=_HS_HEADERS).json()
    with open(f"{LAKEHOUSE_RAW}/{path}", encoding="utf-8") as f:
        return json.load(f)


### Marketing Sources

In [ ]:
sources = _hs_get("hubspot/marketing_sources.json")
df_ms = spark.createDataFrame(sources)
(
    df_ms.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE}_hubspot_marketing_sources")
)
print(f"{BRONZE}_hubspot_marketing_sources: {df_ms.count()} rows")


### Contacts (paginated)

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def _fetch_contacts_page(after: str | None = None) -> dict:
    # PRODUCTION:
    #   params = {"limit": 100, "after": after,
    #             "properties": "firstname,lastname,email,phone,lifecyclestage"}
    #   resp = requests.get(
    #       "https://api.hubapi.com/crm/v3/objects/contacts",
    #       headers=_HS_HEADERS, params=params)
    #   resp.raise_for_status()
    #   return resp.json()
    if after is not None:
        return {"results": [], "paging": None}
    return {"results": _hs_get("hubspot/contacts.json"), "paging": None}


def _contacts_pages() -> Iterator[list[dict]]:
    after: str | None = None
    while True:
        page = _fetch_contacts_page(after)
        results = page.get("results", [])
        if not results:
            break
        yield results
        paging = page.get("paging")
        if not paging:
            break
        after = paging["next"]["after"]


all_contacts = [row for page in _contacts_pages() for row in page]
df_contacts = spark.createDataFrame(all_contacts)
(
    df_contacts.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE}_hubspot_contacts")
)
print(f"{BRONZE}_hubspot_contacts: {df_contacts.count()} rows")


### Activities (paginated)

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def _fetch_activities_page(after: str | None = None) -> dict:
    # PRODUCTION:
    #   params = {"limit": 100, "after": after}
    #   resp = requests.get(
    #       "https://api.hubapi.com/crm/v3/objects/engagements",
    #       headers=_HS_HEADERS, params=params)
    #   resp.raise_for_status()
    #   return resp.json()
    if after is not None:
        return {"results": [], "paging": None}
    return {"results": _hs_get("hubspot/activities.json"), "paging": None}


def _activities_pages() -> Iterator[list[dict]]:
    after: str | None = None
    while True:
        page = _fetch_activities_page(after)
        results = page.get("results", [])
        if not results:
            break
        yield results
        paging = page.get("paging")
        if not paging:
            break
        after = paging["next"]["after"]


all_activities = [row for page in _activities_pages() for row in page]
df_activities = spark.createDataFrame(all_activities)
(
    df_activities.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE}_hubspot_activities")
)
print(f"{BRONZE}_hubspot_activities: {df_activities.count()} rows")


## Dynamics 365 and Paradigm — Copy Jobs

These sources are ingested by **Copy Job** activities in the Data Pipeline, not this notebook.

| Copy Job | Source path | Bronze table |
|---|---|---|
| D365 Courses | `Files/raw/dynamics365/courses.csv` | `bronze_dynamics365_courses` |
| D365 Intakes | `Files/raw/dynamics365/intakes.csv` | `bronze_dynamics365_intakes` |
| D365 Students | `Files/raw/dynamics365/students.csv` | `bronze_dynamics365_students` |
| D365 Enrolments | `Files/raw/dynamics365/enrolments.csv` | `bronze_dynamics365_enrolments` |
| Paradigm Students | `Files/raw/paradigm/students.csv` | `bronze_paradigm_students` |
| Paradigm Units | `Files/raw/paradigm/units.csv` | `bronze_paradigm_units` |
| Paradigm Results | `Files/raw/paradigm/results.csv` | `bronze_paradigm_results` |

In production: D365 uses the Dataverse connector; Paradigm uses scheduled SFTP/blob exports.

## Verification

In [ ]:
bronze_tables = [
    "bronze_hubspot_marketing_sources",
    "bronze_hubspot_contacts",
    "bronze_hubspot_activities",
    "bronze_dynamics365_courses",
    "bronze_dynamics365_intakes",
    "bronze_dynamics365_students",
    "bronze_dynamics365_enrolments",
    "bronze_paradigm_students",
    "bronze_paradigm_units",
    "bronze_paradigm_results",
]

print(f"{'Table':<45} {'Rows':>6}")
print("-" * 53)
for tbl in bronze_tables:
    try:
        n = spark.table(tbl).count()
        print(f"{tbl:<45} {n:>6}")
    except Exception as e:
        print(f"{tbl:<45} ERROR: {e}")
